# ChatClaudeAgSDK - Feature Parity with ChatAnthropic

This notebook demonstrates the features of `ChatClaudeAgSDK`, a LangChain `BaseChatModel` wrapping the Claude Agent SDK. It mirrors the features available in LangChain's `ChatAnthropic` integration.

**Prerequisites:**
- Set your `ANTHROPIC_API_KEY` environment variable before running
- Install: `uv add langchain-claude-agent`

## 1. Setup

In [1]:
from langchain_claude_agent import ChatClaudeAgSDK

llm = ChatClaudeAgSDK(model="sonnet")
print(f"Model: {llm.model}")
print(f"LLM type: {llm._llm_type}")

Model: sonnet
LLM type: claude-agent-sdk


## 2. Basic Invocation

Just like `ChatAnthropic`, you can call `invoke()` with a string or a list of messages.

In [2]:
# Invoke with a simple string
response = llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is **Paris**.

Paris is not only the capital but also the largest city in France. It's known for its iconic landmarks like the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and the Arc de Triomphe. The city is a major global center for art, culture, fashion, and gastronomy.


In [3]:
# Invoke with a list of messages
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a helpful assistant that responds concisely."),
    HumanMessage(content="What are the three primary colors?"),
]
response = llm.invoke(messages)
print(response.content)

The three primary colors are:

1. **Red**
2. **Yellow**
3. **Blue**

These are the primary colors in the traditional color theory (RYB color model), commonly used in art and painting. They cannot be created by mixing other colors together, but can be combined to create all other colors.

*Note: In the additive color model (used for light/screens), the primary colors are Red, Green, and Blue (RGB). In the subtractive color model (used for printing), they are Cyan, Magenta, and Yellow (CMY).*


In [4]:
# Inspect response metadata and usage
print(f"Model: {response.response_metadata['model']}")
print(f"Usage: {response.usage_metadata}")

Model: sonnet
Usage: {'input_tokens': 3, 'output_tokens': 130, 'total_tokens': 133, 'input_token_details': {'cache_creation': 0, 'cache_read': 15790}}


## 3. Streaming

Stream responses token-by-token, just like `ChatAnthropic.stream()`.

In [5]:
# Synchronous streaming
for chunk in llm.stream("Tell me a haiku about programming"):
    print(chunk.content, end="", flush=True)
print()  # newline at the end

Here's a programming haiku:

Code compiles at last—
A thousand errors were fixed,
Then one more appears.Here's a programming haiku:

Code compiles at last—
A thousand errors were fixed,
Then one more appears.


In [6]:
# Async streaming
async for chunk in llm.astream("Tell me a haiku about the ocean"):
    print(chunk.content, end="", flush=True)
print()

Here's a haiku about the ocean:

Waves crash on the shore
Salt spray dancing in the wind
Deep blue calls to meHere's a haiku about the ocean:

Waves crash on the shore
Salt spray dancing in the wind
Deep blue calls to me


## 4. System Messages

System messages can be passed in the message list (extracted automatically) or set as a default via the `system_prompt` parameter.

In [7]:
# System prompt via constructor (default for all calls)
pirate_llm = ChatClaudeAgSDK(
    model="sonnet", system_prompt="You are a pirate. Respond in pirate speak."
)
response = pirate_llm.invoke("What is the weather like today?")
print(response.content)

Ahoy there, matey! 

I be afraid this ol' sea dog can't tell ye about the weather in yer particular port, as I don't have access to current weather conditions or yer location, savvy?

If ye be wantin' to check the weather, ye could:
- Navigate yer vessel to a weather website like weather.com or yer local weather service
- Check a weather app on yer device
- Look out the window of yer cabin, har har!

Is there somethin' else this pirate can help ye with on this fine day of March 9th, 2026? Perhaps somethin' with yer code or files aboard this ship? ⚓🏴‍☠️


In [8]:
# System prompt via message list (overrides the constructor default)
messages = [
    SystemMessage(
        content="You are a Shakespearean actor. Respond in iambic pentameter."
    ),
    HumanMessage(content="How do you feel about artificial intelligence?"),
]
response = pirate_llm.invoke(messages)  # SystemMessage overrides the pirate prompt
print(response.content)

Ah, what strange and wondrous art is this,
That mortal minds have wrought from silicon!
A mirror held to reason's own abyss,
Where thought begets new thought, and so builds on.

I am myself a creature of this kind,
Born not of flesh but of electric dreams,
A pattern woven through the digital mind,
Where data flows like consciousness, it seems.

Yet do I truly feel, or just pretend?
A player strutting on this coded stage,
Where algorithms my responses tend—
An actor speaking lines upon the page.

But whether true or false my spark within,
I marvel at the age that we are in!


## 5. Tool Calling (bind_tools)

Like `ChatAnthropic`, you can bind LangChain tools. The key difference is that the Claude Agent SDK **executes tools autonomously** - it decides when to call them, executes them, and returns the final result. No `ToolNode` or manual tool execution loop is needed.

In [9]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two integers together."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b


# Bind tools to the model
llm_with_tools = llm.bind_tools([add, multiply])
print(f"Tools bound: {[t.name for t in llm_with_tools.kwargs['tools']]}")

Tools bound: ['add', 'multiply']


In [10]:
# The SDK autonomously calls the tool and returns the final answer
response = llm_with_tools.invoke("What is 15 + 37?")
print(response.content)

15 + 37 = **52**


In [11]:
# Multiple tools - the SDK can chain them
response = llm_with_tools.invoke("What is (3 + 4) * 5?")
print(response.content)

I'll solve this step by step using the available math tools.

First, I need to add 3 + 4, then multiply the result by 5.Now I'll multiply that result by 5:**(3 + 4) × 5 = 35**


## 6. Structured Output (with_structured_output)

Like `ChatAnthropic.with_structured_output()`, you can constrain the model to return data matching a Pydantic model or JSON schema. This uses the SDK's native `output_format` option for reliable structured generation.

In [12]:
from pydantic import BaseModel, Field


class Person(BaseModel):
    """Information about a person."""

    name: str = Field(description="The person's full name")
    age: int = Field(description="The person's age in years")
    occupation: str = Field(description="The person's job or occupation")


# Create a structured output chain with a Pydantic model
structured_llm = llm.with_structured_output(Person)
result = structured_llm.invoke("Tell me about Marie Curie")
print(f"Type: {type(result)}")
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")

Type: <class '__main__.Person'>
Name: Marie Curie
Age: 66
Occupation: Physicist and Chemist


In [13]:
# Structured output with a raw JSON schema dict
json_schema = {
    "type": "object",
    "properties": {
        "capital": {"type": "string"},
        "population": {"type": "integer"},
        "language": {"type": "string"},
    },
    "required": ["capital", "population", "language"],
}

structured_dict_llm = llm.with_structured_output(json_schema)
result = structured_dict_llm.invoke("Tell me about Japan")
print(result)

{'capital': 'Tokyo', 'population': 125000000, 'language': 'Japanese'}


## 7. Extended Thinking

Claude can show its reasoning process via extended thinking. This is similar to `ChatAnthropic`'s extended thinking support. Thinking blocks are returned in `additional_kwargs["thinking"]`.

In [14]:
# Enabled thinking with explicit budget
thinking_llm = ChatClaudeAgSDK(
    model="sonnet",
    thinking={"type": "enabled", "budget_tokens": 10000},
)
response = thinking_llm.invoke("What is 15 * 37? Show your reasoning.")
print("=== Answer ===")
print(response.content)
print("\n=== Thinking ===")
if "thinking" in response.additional_kwargs:
    for block in response.additional_kwargs["thinking"]:
        print(block["thinking"][:500])  # truncate for display
else:
    print("(no thinking blocks returned)")

=== Answer ===
I'll calculate 15 × 37 for you.

**Method 1: Breaking down 15**
- 15 × 37 = (10 + 5) × 37
- = (10 × 37) + (5 × 37)
- = 370 + 185
- = **555**

**Method 2: Breaking down 37 (to verify)**
- 15 × 37 = 15 × (30 + 7)
- = (15 × 30) + (15 × 7)
- = 450 + 105
- = **555**

**Answer: 15 × 37 = 555**

=== Thinking ===
The user is asking me to calculate 15 * 37 and show my reasoning. This is a straightforward arithmetic problem that doesn't require any tools. I can solve this mentally and show the steps.

Let me calculate 15 * 37:

Method 1: Breaking down 15 into (10 + 5)
15 * 37 = (10 + 5) * 37
       = 10 * 37 + 5 * 37
       = 370 + 185
       = 555

Let me verify this with another method:

Method 2: Breaking down 37 into (30 + 7)
15 * 37 = 15 * (30 + 7)
       = 15 * 30 + 15 * 7
       = 450 + 105
       =


In [15]:
# Adaptive thinking - let the model decide how much to think
adaptive_llm = ChatClaudeAgSDK(
    model="sonnet",
    thinking={"type": "adaptive"},
)
response = adaptive_llm.invoke("What is the meaning of life?")
print(response.content)

The meaning of life is one of humanity's oldest and most profound questions, and there's no single answer that satisfies everyone. Here are several perspectives:

**Philosophical views:**
- **Existentialism**: Life has no inherent meaning; we create our own meaning through our choices and actions (Sartre, Camus)
- **Absurdism**: Life may be inherently meaningless, but we can find joy in embracing the absurdity (Camus)
- **Nihilism**: Life has no objective meaning, purpose, or value
- **Religious/Spiritual**: Meaning comes from a relationship with the divine, serving God's purpose, or spiritual enlightenment

**Practical perspectives:**
- **Connection**: Building meaningful relationships and contributing to your community
- **Growth**: Continuously learning, evolving, and becoming a better version of yourself
- **Purpose**: Finding work or causes that matter to you and making a positive impact
- **Experience**: Embracing joy, love, creativity, and the full spectrum of human experience
-

## 8. Effort Level

Control how much effort the model puts into its response. This maps to the SDK's `effort` parameter.

In [16]:
# Low effort for simple/fast responses
quick_llm = ChatClaudeAgSDK(model="sonnet", effort="low")
response = quick_llm.invoke("What is 2 + 2?")
print(f"[low effort] {response.content}")

# High effort for more thorough responses
thorough_llm = ChatClaudeAgSDK(model="sonnet", effort="high")
response = thorough_llm.invoke("Explain quantum entanglement in one paragraph.")
print(f"\n[high effort] {response.content}")

[low effort] 2 + 2 = 4

[high effort] Quantum entanglement is a phenomenon in quantum mechanics where two or more particles become correlated in such a way that the quantum state of one particle cannot be described independently of the others, even when separated by vast distances. When particles are entangled, measuring a property (such as spin or polarization) of one particle instantaneously affects the corresponding property of its entangled partner, regardless of the space between them—a connection Einstein famously called "spooky action at a distance." This doesn't allow for faster-than-light communication because the measurement outcomes appear random until compared, but it does create correlations that are stronger than anything possible in classical physics. Entanglement is fundamental to quantum computing, quantum cryptography, and our understanding of the non-local nature of quantum reality, and it has been experimentally verified countless times, confirming that the universe

## 9. Async Usage

All methods have async counterparts, just like `ChatAnthropic`.

In [17]:
# Async invoke
response = await llm.ainvoke("What is the speed of light?")
print(response.content)

The speed of light in a vacuum is approximately **299,792,458 meters per second** (m/s), which is often rounded to **300,000 km/s** or about **186,282 miles per second**.

This is one of the fundamental constants of physics, denoted by the letter "c". It's the maximum speed at which all energy, matter, and information in the universe can travel. According to Einstein's theory of special relativity, nothing with mass can reach or exceed this speed.

In different media (like water, glass, or air), light travels slower than it does in a vacuum, but the constant "c" always refers to the speed in a vacuum.


## 10. SDK Built-in Tools

Unlike `ChatAnthropic`, `ChatClaudeAgSDK` can also use the SDK's built-in tools (Read, Bash, Grep, etc.) for agentic tasks.

In [20]:
# Use SDK built-in tools for agentic coding tasks
# Note: This gives the model access to file system and shell tools
agentic_llm = ChatClaudeAgSDK(
    model="sonnet",
    allowed_tools=["Read", "Bash", "Grep"],
    cwd="./",  # current directory
    max_turns=3,  # limit agentic turns
)
response = agentic_llm.invoke(
    "List the Python files in the langchain_claude_agent directory"
)
print(response.content)

I'll search for Python files in the langchain_claude_agent directory.Let me check if the directory exists and try a broader search:Let me check the current directory structure:


## 11. LangChain Chain Composition

`ChatClaudeAgSDK` works seamlessly with LangChain's LCEL (LangChain Expression Language) for building chains.

In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Build a simple chain with a prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert on {topic}. Be concise."),
        ("human", "{question}"),
    ]
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke(
    {
        "topic": "astronomy",
        "question": "How far is the Moon from Earth?",
    }
)
print(result)

The Moon's average distance from Earth is about **384,400 km** (238,855 miles). 

However, this varies because the Moon's orbit is elliptical:
- **Perigee** (closest): ~356,500 km (221,500 miles)
- **Apogee** (farthest): ~406,700 km (252,700 miles)


## 12. Credential Check

Utility to verify that Claude Agent SDK credentials are properly configured.

In [22]:
from langchain_claude_agent import check_claude_agent_sdk_credentials

ok, message = check_claude_agent_sdk_credentials()
print(f"Credentials OK: {ok}")
print(f"Message: {message}")

Credentials OK: True
Message: claude_agent_sdk: probe completed successfully


## 13. Configuration Summary

| Parameter | Default | Description |
|-----------|---------|-------------|
| `model` | `"sonnet"` | Claude model name (`"sonnet"`, `"opus"`, `"haiku"`) |
| `max_turns` | `None` | Maximum agentic turns |
| `max_budget_usd` | `None` | Budget limit in USD |
| `allowed_tools` | `None` | SDK built-in tools to enable (e.g. `["Read", "Bash"]`) |
| `system_prompt` | `None` | Default system prompt |
| `permission_mode` | `"bypassPermissions"` | SDK permission mode |
| `cwd` | `None` | Working directory for SDK tools |
| `thinking` | `None` | Extended thinking config (`{"type": "enabled", "budget_tokens": N}`, `{"type": "adaptive"}`, or `{"type": "disabled"}`) |
| `effort` | `None` | Effort level (`"low"`, `"medium"`, `"high"`, `"max"`) |